# 2D calibration: symmetric windows

The default inspects the bundled fits (ids 5–24). Set RUN_CALIBRATION=True to generate and save a new series; then update the ids in the inspection cell. Calibration can take substantially longer than plotting.

**Setup:** from the repository root, run `conda env create -f environment.yml` then `conda activate single-photon-renorm`, then `jupyter lab`. Select that Python environment as the kernel. The first cell also supports running directly from the checkout.

**How to use:** run cells from top to bottom. Parameters are grouped before the simulation or plotting cells. Figures display inline; `SAVE_FIGURES=False` prevents overwriting exported PDFs. Units: ħ = c = 1. See `guide_utilisation.pdf` at the repository root for notation and the calibration workflow.


# Symmetric Frequency Windows

Generate and inspect the 2D bare parameters associated with symmetric windows $[\Omega_A-\lambda,\Omega_A+\lambda]$.

In [ ]:
RUN_CALIBRATION = False
# Run from the repository root or any of its subdirectories.
import sys
from pathlib import Path
project_root = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                     if (p / "src" / "experiment.py").is_file()), None)
if project_root is None:
    raise FileNotFoundError("Open Jupyter inside the single_photon_code checkout; see README.md.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
benchmark_root = project_root / "waveguide_model"
setup_root = project_root / "2d_setup"
SAVE_FIGURES = False  # Set True to export the figures to results/fig.


import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

resol = 180
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "axes.linewidth": 0.7,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
})


from importlib import import_module
bare_param_path = import_module("2d_setup.database").bare_param_path
load_bare_parameters_csv = import_module("2d_setup.database").load_bare_parameters_csv
read_bare_parameters_metadata = import_module("2d_setup.database").read_bare_parameters_metadata
read_target_profile_metadata = import_module("2d_setup.database").read_target_profile_metadata
save_bare_parameters_csv = import_module("2d_setup.database").save_bare_parameters_csv
target_profile_path = import_module("2d_setup.database").target_profile_path
get_2d_bare_parameters = import_module("2d_setup.bare_param").get_2d_bare_parameters
load_geometry_by_id = import_module("2d_setup.geometry").load_geometry_by_id

pi = np.pi


<h2>Load target profile</h2>

In [ ]:
id_target_profile = 1

target_profile_csv = target_profile_path(id_target_profile)
if not target_profile_csv.exists():
    raise FileNotFoundError(f"Missing target_profile_id{id_target_profile}.csv.")

target_metadata = read_target_profile_metadata(target_profile_csv)
geometry = load_geometry_by_id(target_metadata["id_geom"])
atom_positions = geometry["atom_positions"]
N_A = geometry["N_A"]

Omega_A_tab = target_metadata["Omega_A"]
d_A_tab = target_metadata["d_A"]
if not np.allclose(Omega_A_tab, Omega_A_tab[0]):
    raise ValueError("Symmetric windows require a common atomic frequency Omega_A.")
Omega_A = float(Omega_A_tab[0])

print(f"Target profile: target_profile_id{id_target_profile}.csv")
print(f"Geometry: geometry_id{target_metadata['id_geom']}, N_A={N_A}")
print(f"Omega_A = {Omega_A / pi:.6g} pi")
print(f"Reference window = [{target_metadata['reference_omega_min'] / pi:.6g} pi, {target_metadata['reference_omega_max'] / pi:.6g} pi]")


<h2>Choose symmetric windows</h2>

In [ ]:
# Free parameter: edit this set as needed.
lambda_values = pi * np.linspace(2, 9.9, 20)

windows = [
    {
        "lambda": float(lbda),
        "Omega_min": Omega_A - float(lbda),
        "Omega_max": Omega_A + float(lbda),
    }
    for lbda in lambda_values
]

pd.DataFrame(
    {
        "lambda/pi": [window["lambda"] / pi for window in windows],
        "ir_cutoff/pi": [window["Omega_min"] / pi for window in windows],
        "uv_cutoff/pi": [window["Omega_max"] / pi for window in windows],
    }
)


<h2>Generate bare parameters</h2>

In [ ]:
if RUN_CALIBRATION:
    generated_rows = []

    for window in tqdm(windows, desc="symmetric windows"):
        cutoffs = {
            "Omega_min": window["Omega_min"],
            "Omega_max": window["Omega_max"],
        }
        Omega_0_tab, d_0_tab = get_2d_bare_parameters(
            target_profile_csv,
            atom_positions,
            cutoffs,
        )

        output_path = bare_param_path()
        save_bare_parameters_csv(Omega_0_tab, d_0_tab, target_metadata, cutoffs, output_path)
        bare_param_id = int(output_path.stem.removeprefix("bare_param_id"))

        generated_rows.append(
            {
                "bare_param_id": bare_param_id,
                "lambda_pi": window["lambda"] / pi,
                "ir_cutoff_pi": cutoffs["Omega_min"] / pi,
                "uv_cutoff_pi": cutoffs["Omega_max"] / pi,
                "path": output_path,
            }
        )

    generated = pd.DataFrame(generated_rows)
    generated
else:
    print("Inspecting bundled calibration ids; fitting skipped.")


<h2>Load saved bare parameters</h2>

In [ ]:
# Free parameters: choose the saved ids to inspect.
bare_param_id_start = 5
bare_param_id_stop = 24
bare_param_ids = range(bare_param_id_start, bare_param_id_stop + 1)

loaded_rows = []
for bare_param_id in bare_param_ids:
    path = bare_param_path(bare_param_id)
    if not path.exists():
        raise FileNotFoundError(f"Missing bare_param_id{bare_param_id}.csv.")

    metadata = read_bare_parameters_metadata(path)
    if metadata["id_target_profile"] != id_target_profile:
        raise ValueError(
            f"bare_param_id{bare_param_id} was generated from "
            f"target_profile_id{metadata['id_target_profile']}, not target_profile_id{id_target_profile}."
        )

    Omega_0_tab, d_0_tab = load_bare_parameters_csv(path)
    lbda = 0.5 * (metadata["Omega_max"] - metadata["Omega_min"])
    center = 0.5 * (metadata["Omega_max"] + metadata["Omega_min"])

    loaded_rows.append(
        {
            "bare_param_id": bare_param_id,
            "lambda": lbda,
            "lambda_pi": lbda / pi,
            "center_pi": center / pi,
            "ir_cutoff_pi": metadata["Omega_min"] / pi,
            "uv_cutoff_pi": metadata["Omega_max"] / pi,
            "Omega_0": Omega_0_tab,
            "d_0": d_0_tab,
            "path": path,
        }
    )

bare_param_scan = pd.DataFrame(loaded_rows).sort_values("lambda")
bare_param_scan[["bare_param_id", "lambda_pi", "center_pi", "ir_cutoff_pi", "uv_cutoff_pi", "path"]]


In [ ]:
lambda_axis = bare_param_scan["lambda"].to_numpy() / pi
Omega_0 = np.vstack(bare_param_scan["Omega_0"].to_numpy())
d_0 = np.vstack(bare_param_scan["d_0"].to_numpy())

fig, (omega_ax, d_ax) = plt.subplots(1, 2, figsize=(10.5, 4), dpi=resol, gridspec_kw={"wspace": 0.35})
colors = plt.cm.viridis(np.linspace(0.08, 0.92, N_A))

for j, color in enumerate(colors):
    omega_ax.plot(lambda_axis, Omega_0[:, j] / pi, marker="o", markersize=3, linewidth=1.0, color=color, label=f"j={j}")
    d_ax.plot(lambda_axis, d_0[:, j], marker="o", markersize=3, linewidth=1.0, color=color, label=f"j={j}")

omega_ax.axhline(Omega_A / pi, color="0.25", linestyle="--", linewidth=0.8)
d_ax.axhline(np.mean(d_A_tab), color="0.25", linestyle="--", linewidth=0.8)

omega_ax.set_title(r"$\Omega_j^{(0)}(\lambda)$", fontweight="bold")
omega_ax.set_xlabel(r"$\lambda / \pi$")
omega_ax.set_ylabel(r"$\Omega_j^{(0)} / \pi$")
omega_ax.grid(color="0.9", linewidth=0.5)

d_ax.set_title(r"$d_{0,j}(\lambda)$", fontweight="bold")
d_ax.set_xlabel(r"$\lambda / \pi$")
d_ax.set_ylabel(r"$d_{0,j}$")
d_ax.grid(color="0.9", linewidth=0.5)

handles, labels = omega_ax.get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, -0.08), ncol=min(N_A, 5), frameon=False)
plt.tight_layout(rect=(0, 0.08, 1, 1))
plt.show()

summary_rows = []
for _, row in bare_param_scan.iterrows():
    for j, (Omega_0, d_value) in enumerate(zip(row["Omega_0"], row["d_0"])):
        summary_rows.append(
            {
                "bare_param_id": row["bare_param_id"],
                "lambda_pi": row["lambda_pi"],
                "j": j,
                "Omega_0_pi": Omega_0 / pi,
                "d_0": d_value,
                "path": row["path"],
            }
        )

bare_param_vs_lambda = pd.DataFrame(summary_rows)
bare_param_vs_lambda.head()
